<a href="https://colab.research.google.com/github/heberdavi/mba-engsoft-tcc/blob/main/notebooks/04_inventario_processamento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Célula 1: Preparação do ambiente
# Instala o Streamlit e o Localtunnel (para gerar o link público)
!pip install -q streamlit
!npm install -q -g localtunnel

# Monta o Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Célula 2
%%writefile app_inventario.py
import streamlit as st
import pandas as pd
import sqlite3
import plotly.express as px

# Configuração da página para aproveitar toda a largura da tela
st.set_page_config(layout="wide", page_title="Auditoria do Inventário de Suporte")

# Função com Cache para performance do pipeline
@st.cache_data
def load_data():
    path = "/content/drive/MyDrive/mba-engsof-tcc/versao_final/data/base-dados-final.db"
    conn = sqlite3.connect(path)

    query = """
      with corpus as (
      select
        gl.id genero_id, gl.nome genero,
        l.nome nome_livro, l.abreviacao abreviacao_livro,
        v.numero_capitulo capitulo, v.numero_verso verso, v.texto texto,
        t.antidoto_referencia topico_classificacao_final,
        vt.similaridade_final topico_score_final,
        case vs.sentimento_num when 0 then 'Neutro' when 1 then 'Positivo' when -1 then 'Negativo' end sentimento_classificacao_final,
        max(vs.score_pos, vs.score_neg, vs.score_neu) sentimento_score_final,
        -- IA EXPLICAVEL --
        vl.texto_limpo, length(v.texto) tamanho_texto,
        case max(vt.p_exaustao, vt.p_transitoriedade, vt.p_vazio)
          when vt.p_exaustao then 'Exaustão vs. Refrigério'
          when vt.p_transitoriedade then 'Transitoriedade vs. Solidez'
          when vt.p_vazio then 'Vazio vs. Propósito'
        end topico_melhor_classificacao_existencial,
        max(vt.p_exaustao, vt.p_transitoriedade, vt.p_vazio) topico_melhor_score_existencial,
        vt.p_exaustao topico_score_exaustao, vt.p_transitoriedade topico_score_transitoriedade, vt.p_vazio topico_score_vazio, vt.p_narrativo topico_score_narrativo,
        vs.score_pos sentimento_score_positivo, vs.score_neg sentimento_score_negativo, vs.score_neu sentimento_score_neutro,

        -- CÁLCULO DO POTENCIAL DE ALÍVIO (PA)
        -- Fórmula baseada na força do melhor eixo existencial multiplicada pela polaridade emocional ativa (1 - score_neutro)
        round(max(vt.p_exaustao, vt.p_transitoriedade, vt.p_vazio) * (1.0 - vs.score_neu), 4) as potencial_alivio,

        vt.margem_dominancia,
        vt.entropia,
        vt.gap_confianca,
        vt.status_decisao decisao_final
      from genero_literario gl
      join livro l on l.genero_id = gl.id
      join verso v on v.livro_id = l.id
      join verso_limpo vl on vl.verso_id = v.id
      join verso_sentimento vs on vs.verso_id = v.id
      join verso_topico vt on vt.verso_id = v.id
      join topico t on t.id = vt.topico_id)
      select genero, nome_livro, abreviacao_livro,
        capitulo, verso, texto,
        topico_classificacao_final, topico_score_final,
        sentimento_classificacao_final, sentimento_score_final,
        tamanho_texto, texto_limpo,
        topico_melhor_classificacao_existencial,
        topico_melhor_score_existencial,
        topico_score_exaustao, topico_score_transitoriedade, topico_score_vazio, topico_score_narrativo,
        margem_dominancia, entropia, gap_confianca,
        sentimento_score_positivo, sentimento_score_negativo, sentimento_score_neutro,
        potencial_alivio,
        decisao_final,
        case
          when tamanho_texto < 35 and margem_dominancia < 0.25
              then 'Texto muito curto'
          else
              case
              when genero_id in (1, 2) and topico_melhor_score_existencial > 0.88 and margem_dominancia > 0.15
            then 'Classificação existencial ('||round(topico_melhor_score_existencial, 2)||') no eixo '||topico_melhor_classificacao_existencial||' superou o score de referência para o gênero (0.88). A margem de dominância ('||round(margem_dominancia, 2)||') também superou a referência para o eixo (0.15)'
              when genero_id in (3, 4) and topico_melhor_score_existencial > 0.50
                  then 'Classificação existencial ('||round(topico_melhor_score_existencial, 2)||') no eixo '||topico_melhor_classificacao_existencial||' superou o score de referência para o gênero (0.50)'
              when genero_id in (5, 6) and (topico_melhor_score_existencial > 0.60 or (topico_melhor_score_existencial > 0.45 and margem_dominancia > 0.10)) then
                  case
                      when topico_melhor_score_existencial > 0.60
                          then 'Classificação existencial ('||round(topico_melhor_score_existencial, 2)||') no eixo '||topico_melhor_classificacao_existencial||' superou o score de referência para o gênero (0.60)'
                      else 'Classificação existencial ('||round(topico_melhor_score_existencial, 2)||') no eixo '||topico_melhor_classificacao_existencial||' superou o score de referência para o gênero (0.45). A margem de dominância ('||round(margem_dominancia, 2)||') também superou a referência para o eixo (0.10)'
                  end
              when genero_id = 7 and topico_melhor_score_existencial > 0.75
                  then 'Classificação existencial ('||round(topico_melhor_score_existencial, 2)||') no eixo '||topico_melhor_classificacao_existencial||' superou o score de referência para o gênero (0.75)'
              else
                  'Regra geral'
              end
          end motivo_decisao
      from corpus
    """
    df = pd.read_sql(query, conn)
    conn.close()
    return df

df = load_data()

# --- INTERFACE STREAMLIT ---
st.title("🛡️ Classificação das Âncoras Emocionais")
st.markdown("Interface técnica para validação de regras de negócio e explicabilidade do modelo (XAI).")

# Sidebar para filtros
with st.sidebar:
    st.header("Painel de Controle")
    busca = st.text_input("Buscar termo no verso (ex: vida, alma)")
    confianca_min = st.slider("Confiança Mínima (Score Eixo)", 0.0, 1.0, 0.0)

    # Novo Filtro: Potencial de Alívio
    alivio_min = st.slider("Potencial de Alívio Mínimo", 0.0, 1.0, 0.0)

    sentimento_opcoes = st.multiselect("Filtrar Sentimento", options=sorted(df['sentimento_classificacao_final'].unique()))
    genero_list = st.multiselect("Filtrar Gênero", options=sorted(df['genero'].unique()))
    topico_list = st.multiselect("Filtrar Eixo", options=sorted(df['topico_classificacao_final'].unique()))
    status_list = st.multiselect("Decisão Final", options=sorted(df['decisao_final'].unique()))

# Aplicação dos Filtros
df_view = df.copy()
if busca:
    df_view = df_view[df_view['texto'].str.contains(busca, case=False, na=False)]
if confianca_min > 0:
    df_view = df_view[df_view['topico_melhor_score_existencial'] >= confianca_min]
if alivio_min > 0:
    df_view = df_view[df_view['potencial_alivio'] >= alivio_min]
if sentimento_opcoes:
    df_view = df_view[df_view['sentimento_classificacao_final'].isin(sentimento_opcoes)]
if genero_list:
    df_view = df_view[df_view['genero'].isin(genero_list)]
if topico_list:
    df_view = df_view[df_view['topico_classificacao_final'].isin(topico_list)]
if status_list:
    df_view = df_view[df_view['decisao_final'].isin(status_list)]

# Tabela de Dados Interativa
st.subheader(f"Registros Encontrados: {len(df_view)}")
st.dataframe(df_view, use_container_width=True)

# Inspeção Detalhada com Validação de Dados
st.divider()
st.subheader("🔍 Inspeção de IA Explicável (XAI)")

if not df_view.empty:
    df_view['label_auditoria'] = (
        df_view['abreviacao_livro'] + " " +
        df_view['capitulo'].astype(str) + ":" +
        df_view['verso'].astype(str) + " - " +
        df_view['texto'].str[:50] + "..."
    )

    selected_label = st.selectbox(
        "Selecione o registro para detalhamento:",
        options=df_view['label_auditoria'].tolist(),
        key="auditoria_selector"
    )

    item_match = df_view[df_view['label_auditoria'] == selected_label]

    if not item_match.empty:
        item = item_match.iloc[0]
        c1, c2 = st.columns([1, 1])

        with c1:
            st.markdown(f"**📖 Endereço:** {item['nome_livro']} ({item['abreviacao_livro']}) {item['verso']}")
            st.markdown(f"**🎭 Gênero:** {item['genero']}")
            st.info(f"**Texto:** {item['texto']}")

            # Visualização do Potencial de Alívio calculada no pipeline
            pa_score = float(item['potencial_alivio'])
            if pa_score >= 0.7:
                st.metric(label="✨ Potencial de Alívio", value=f"{pa_score:.2f}", delta="Alto Impacto")
            elif pa_score >= 0.4:
                st.metric(label="✨ Potencial de Alívio", value=f"{pa_score:.2f}", delta="Médio Impacto", delta_color="off")
            else:
                st.metric(label="✨ Potencial de Alívio", value=f"{pa_score:.2f}", delta="Baixo Impacto/Narrativo", delta_color="inverse")

            if "Regra geral" in str(item['motivo_decisao']) or "muito curto" in str(item['motivo_decisao']):
                st.error(f"**Justificativa:** {item['motivo_decisao']}")
            else:
                st.success(f"**Justificativa:** {item['motivo_decisao']}")

        with c2:
            chart_data = pd.DataFrame({
                'Eixo Existencial': ['Exaustão vs. Refrigério', 'Transitoriedade vs. Solidez', 'Vazio vs. Propósito', 'Narrativo/Normativo'],
                'Score (0-1)': [
                    float(item['topico_score_exaustao']),
                    float(item['topico_score_transitoriedade']),
                    float(item['topico_score_vazio']),
                    float(item['topico_score_narrativo'])
                ]
            })

            fig = px.bar(
                chart_data,
                x='Score (0-1)',
                y='Eixo Existencial',
                orientation='h',
                title="Distribuição de Probabilidades (Softmax)",
                range_x=[0,1],
                color='Score (0-1)',
                color_continuous_scale='Blues'
            )
            fig.update_layout(showlegend=False, height=350, margin=dict(l=20, r=20, t=50, b=20))
            st.plotly_chart(fig, use_container_width=True)
    else:
        st.info("🔄 Atualizando visualização...")
else:
    st.warning("⚠️ Nenhum registro encontrado para os filtros selecionados na barra lateral.")

In [ ]:
# Célula 3: Execução e túnel de acesso (Versão Estabilizada)
!pip install -q pyngrok

import os
import time
from pyngrok import ngrok
from google.colab import userdata

# 1. Limpeza rigorosa de processos órfãos
# Encerramos o ngrok e o streamlit para garantir que a porta 8501 seja liberada
ngrok.kill()
!pkill ngrok
!pkill streamlit

# 2. Configuração do Token
NGROK_TOKEN = userdata.get('NGROK_TOKEN')
ngrok.set_auth_token(NGROK_TOKEN)

# 3. Execução do Streamlit em background
# Redirecionamos a saída para 'streamlit.log' para diagnóstico se algo falhar
print("Iniciando o servidor Streamlit...")
!nohup streamlit run app_inventario.py > streamlit.log 2>&1 &

# --- AJUSTE CRUCIAL: PAUSA PARA INICIALIZAÇÃO ---
# Aguardamos o Streamlit carregar completamente antes de abrir o túnel
time.sleep(8)

# 4. Abertura do Túnel com tratamento de exceção
try:
    # Abrimos o túnel na porta padrão do Streamlit
    public_url = ngrok.connect(8501, name="inventario_auditoria_v3")
    print(f"\n✅ SUCESSO! Inventário online.")
    print(f"🔗 Clique aqui para abrir: {public_url}")

except Exception as e:
    print(f"\n❌ Erro ao conectar o túnel: {e}")
    print("\n--- Verificação de Diagnóstico ---")
    # Se falhar, mostramos as últimas linhas do log do Streamlit para entender o porquê
    if os.path.exists("streamlit.log"):
        with open("streamlit.log", "r") as f:
            print("Log do Streamlit:", f.readlines()[-5:])
    print("\n💡 DICA: Se o erro for 'connection refused', tente rodar esta célula novamente.")